# End-to-End fMRI Music Genre Encoding Pipeline

This notebook reconstructs the exact workflow developed for the encoding models, tracking the transformation of auditory stimuli into predicted Blood-Oxygen-Level-Dependent (BOLD) responses natively.

## Extracted Features Summary Table

The dataset integrates distinct categorical and semantic arrays extracted from sequential audio runs. Below summarizes their representation limits prior to temporal HRF scaling.

| Feature Model | Dimensions | Description |
|---------------|------------|-------------|
| **Cochlear**  | 128        | Mel filterbank approximating peripheral limits, pooled temporally locally via $1.5s$ blocks. |
| **MTF**       | 302 PCA    | Modulation Temporal Functions across 10 temporal and spectral limits mapped to 20 sub-bands. Original 2000 vectors reduced algorithmically predicting major variances. |
| **MFCC**      | 12         | Standard Mel-Frequency Cepstral Coefficients summarizing timbral textures dynamically at bandwidth parameters. |


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import nibabel as nib
import matplotlib.pyplot as plt
from tqdm import tqdm
from nilearn.masking import compute_epi_mask
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# Workspace Paths
import os
ROOT = os.path.dirname(os.getcwd())  # run from notebooks/, ROOT = repo root
AUDIO_DIR = os.path.join(ROOT, "recons_audio")
DATASET_DIR = os.environ.get("DS003720_DIR", os.path.join(ROOT, "..", "datasets", "ds003720-download"))
FMRI_DIR  = os.path.join(DATASET_DIR, "sub-001", "func")
FEAT_DIR  = os.path.join(ROOT, "features")
os.makedirs(FEAT_DIR, exist_ok=True)

TRAIN_RUNS = [f"sub-001_task-Training_run-{i:02d}" for i in range(1, 13)]
TEST_RUNS = [f"sub-001_task-Test_run-{i:02d}" for i in range(1, 7)]

# Parameters
SR = 22050
TR = 1.5
HOP_LEN = int(0.010 * SR)   # 10 ms
WIN_LEN = int(0.025 * SR)   # 25 ms
N_FILTERS = 128


--- 
## 1. Feature Extraction Pipelines
Below are the foundational logic functions dictating explicit frequency and mathematical parameters utilized.

In [ ]:
def downsample_to_TR(feat):
    frames_per_TR = int(TR / (HOP_LEN / SR))
    T = feat.shape[1] // frames_per_TR
    out = [feat[:, i*frames_per_TR:(i+1)*frames_per_TR].mean(axis=1) for i in range(T)]
    return np.array(out)

def compute_cochleogram(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=WIN_LEN, hop_length=HOP_LEN, n_mels=N_FILTERS, fmin=100, fmax=8000, power=2.0)
    return np.log1p(S)

def compute_mtf(cochleo):
    freq_bins, time_bins = cochleo.shape
    omega = np.array([2.8, 4.0, 5.7, 8.0, 11.3, 16.0, 22.6, 32.0, 45.3, 64.0])
    Omega = np.array([0.35, 0.50, 0.71, 1.0, 1.41, 2.0, 2.83, 4.0, 5.66, 8.0])
    
    mtf_features = []
    for w in omega:
        fft_time = np.fft.fft(cochleo, axis=1)
        freqs = np.fft.fftfreq(time_bins, d=HOP_LEN/SR)
        filt = fft_time * (np.abs(freqs - w) < w*0.2)
        temp_filtered = np.real(np.fft.ifft(filt, axis=1))
        
        for W in Omega:
            fft_freq = np.fft.fft(temp_filtered, axis=0)
            freqs_f = np.fft.fftfreq(freq_bins)
            filt_f = fft_freq * (np.abs(freqs_f - W) < W*0.2)[:, None]
            spec_filtered = np.real(np.fft.ifft(filt_f, axis=0))
            
            energy = spec_filtered**2
            energy_bands = np.array([np.mean(b, axis=0) for b in np.array_split(energy, 20, axis=0)])
            mtf_features.append(energy_bands)
            
    mtf_features = np.concatenate(mtf_features, axis=0)
    return np.log1p(mtf_features)

def compute_mfcc(y):
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=WIN_LEN, hop_length=HOP_LEN, n_mels=N_FILTERS, fmin=100, fmax=8000, power=2.0)
    return librosa.feature.mfcc(S=librosa.power_to_db(S), n_mfcc=12)


--- 
## 2. Feature Visualization
Let's visually inspect how the distinct feature pipelines represent auditory blocks natively prior to pooling.

In [ ]:
sample_audio = os.path.join(AUDIO_DIR, "sub-001_task-Test_run-01_audio.wav")
if os.path.exists(sample_audio):
    print("Loading sample audio block for feature visualizations...")
    y, sr = librosa.load(sample_audio, sr=SR, duration=5.0) # Load 5 seconds sample
    
    coch_samp = compute_cochleogram(y)
    mtf_samp = compute_mtf(coch_samp)
    mfcc_samp = compute_mfcc(y)
    
    fig, ax = plt.subplots(3, 1, figsize=(10, 12))
    
    # Plot Cochlear
    img1 = librosa.display.specshow(coch_samp, x_axis='time', sr=SR, hop_length=HOP_LEN, ax=ax[0], cmap='magma')
    ax[0].set_title('Cochlear Log-Spectrogram (128 Channels)')
    fig.colorbar(img1, ax=ax[0], format="%+2.f")
    
    # Plot MTF 
    img2 = librosa.display.specshow(mtf_samp, x_axis='time', sr=SR, hop_length=HOP_LEN, ax=ax[1], cmap='plasma')
    ax[1].set_title('MTF Extracted Feature Profile (2000 Dimensions - Sample Preview)')
    fig.colorbar(img2, ax=ax[1], format="%+2.f")
    
    # Plot MFCC
    img3 = librosa.display.specshow(mfcc_samp, x_axis='time', sr=SR, hop_length=HOP_LEN, ax=ax[2], cmap='viridis')
    ax[2].set_title('MFCC Feature Textures (12 Channels)')
    fig.colorbar(img3, ax=ax[2], format="%+2.f")
    
    plt.tight_layout()
    plt.show()
else:
    print("Sample audio not found locally.")


--- 
## 3. Data Transformations & Extraction Iteration
This segment applies PCA matrix thresholds, builds HRF BOLD delays uniformly, and extracts the runs natively into arrays utilizing local storage.

In [ ]:
def extract_all_features():
    audio_files = sorted([f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")])
    for f in audio_files:
        out_path = os.path.join(FEAT_DIR, f.replace(".wav", "_features.npz"))
        if os.path.exists(out_path): continue
            
        print(f"Extracting bounds for {f}...")
        y, _ = librosa.load(os.path.join(AUDIO_DIR, f), sr=SR)
        cochleo = compute_cochleogram(y)
        np.savez_compressed(out_path,
                            cochlear=downsample_to_TR(cochleo),
                            mtf=downsample_to_TR(compute_mtf(cochleo)),
                            mfcc=downsample_to_TR(compute_mfcc(y)))

def apply_hrf_delay(F, delays=[1, 2, 3, 4, 5]):
    T, N = F.shape
    return np.concatenate([np.pad(F, ((d,0), (0,0)))[:-d] if d<T else np.zeros_like(F) for d in delays], axis=1)

def aggregate_features(runs, pca_model=None):
    c_list, m_list, mf_list = [], [], []
    for r in runs:
        data = np.load(os.path.join(FEAT_DIR, f"{r}_audio_features.npz"))
        c_list.append(data['cochlear']); m_list.append(data['mtf']); mf_list.append(data['mfcc'])
        
    Coch = StandardScaler().fit_transform(np.concatenate(c_list, axis=0))
    Mtf = StandardScaler().fit_transform(np.concatenate(m_list, axis=0))
    Mfcc = StandardScaler().fit_transform(np.concatenate(mf_list, axis=0))
    
    if pca_model is None:
        pca_model = PCA(n_components=302)
        Mtf = pca_model.fit_transform(Mtf)
    else:
        Mtf = pca_model.transform(Mtf)
        
    return apply_hrf_delay(Coch), apply_hrf_delay(Mtf), apply_hrf_delay(Mfcc), pca_model

print("Verifying feature derivations exist locally... (Skipped if cached)")
extract_all_features()

print("Aggregating Training constraints...")
F_coch_tr, F_mtf_tr, F_mfcc_tr, pca_model = aggregate_features(TRAIN_RUNS)
print("Aggregating Testing constraints...")
F_coch_ts, F_mtf_ts, F_mfcc_ts, _ = aggregate_features(TEST_RUNS, pca_model)


--- 
## 4. Chunked Ridge Regressions & Model Scaling
Given the BOLD fMRI dataset spans ~178,000 spatial features mapping continuous dimensions natively, standard estimators easily break physical constraints (RAM Limits). Here we compute regression natively via blocks bypassing memory bottlenecks exclusively.

In [ ]:
def get_mask():
    sample_path = os.path.join(FMRI_DIR, f"{TRAIN_RUNS[0]}_bold.nii")
    mask = compute_epi_mask(nib.load(sample_path)).get_fdata(dtype=np.float32) > 0
    return mask

def execute_batched_ridge(F_tr, F_ts, mask_data, model_name, alpha=1000.0):
    print(f"\n===> Training Native Ridge for {model_name}")
    N, V = F_tr.shape[1], mask_data.sum()
    XTX = np.zeros((N, N), dtype=np.float32)
    XTY = np.zeros((N, V), dtype=np.float32)
    
    start = 0
    for r in TRAIN_RUNS:
        img = nib.load(os.path.join(FMRI_DIR, f"{r}_bold.nii")).get_fdata(dtype=np.float32)
        Y_run = img[mask_data].T; del img; gc.collect()
        Y_run = Y_run[:400] # Temporal bounds match fixed audio arrays
        
        std = Y_run.std(axis=0); std[std==0] = 1.0
        Y_run = (Y_run - Y_run.mean(axis=0)) / std
        
        XTX += F_tr[start:start+400].T @ F_tr[start:start+400]
        XTY += F_tr[start:start+400].T @ Y_run
        start += 400
        
    XTX += alpha * np.eye(N)
    W = np.zeros_like(XTY)
    for v in range(0, V, 20000):
        W[:, v:v+20000] = np.linalg.solve(XTX, XTY[:, v:v+20000])
        
    print(f"Evaluations active on {TEST_RUNS[0]}...{TEST_RUNS[-1]}")
    sum_y, sum_y2, sum_yP, sum_yP2, sum_cross = (np.zeros(V, dtype=np.float32) for _ in range(5))
    start, total_N = 0, 0
    
    for r in TEST_RUNS:
        img = nib.load(os.path.join(FMRI_DIR, f"{r}_bold.nii")).get_fdata(dtype=np.float32)
        Y_run = img[mask_data].T; del img; gc.collect()
        Y_run = Y_run[:400]; total_N += 400
        
        std = Y_run.std(axis=0); std[std==0] = 1.0
        Y_run = (Y_run - Y_run.mean(axis=0)) / std
        
        Y_pred = F_ts[start:start+400] @ W
        sum_y += Y_run.sum(axis=0); sum_y2 += (Y_run**2).sum(axis=0)
        sum_yP += Y_pred.sum(axis=0); sum_yP2 += (Y_pred**2).sum(axis=0)
        sum_cross += (Y_run * Y_pred).sum(axis=0)
        start += 400
        
    mean_y, mean_yP = sum_y/total_N, sum_yP/total_N
    var_y, var_yP = sum_y2 - total_N*(mean_y**2), sum_yP2 - total_N*(mean_yP**2)
    denom = np.sqrt(var_y * var_yP)
    
    corrs = np.zeros(V, dtype=np.float32)
    valid = denom > 0
    corrs[valid] = (sum_cross[valid] - total_N*mean_y[valid]*mean_yP[valid]) / denom[valid]
    print(f"=> Resulting Mean Pearson Score: {np.mean(corrs):.4f}")
    return corrs
    
mask_data = get_mask()
corrs_coch = execute_batched_ridge(F_coch_tr, F_coch_ts, mask_data, "Cochlear")
corrs_mtf = execute_batched_ridge(F_mtf_tr, F_mtf_ts, mask_data, "MTF (PCA=302)")
corrs_mfcc = execute_batched_ridge(F_mfcc_tr, F_mfcc_ts, mask_data, "MFCC")


--- 
## 5. Model Feature Effectiveness Comparisons
The metrics demonstrate exactly how much representational capacity exists between baseline auditory extractions and specific categorical/modulation architectures.

In [ ]:
# Filter isolated regions evaluating only top sensory response bounds natively
top_n = 1000
c_top = np.sort(corrs_coch[~np.isnan(corrs_coch)])[-top_n:]
m_top = np.sort(corrs_mtf[~np.isnan(corrs_mtf)])[-top_n:]
mf_top = np.sort(corrs_mfcc[~np.isnan(corrs_mfcc)])[-top_n:]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(['Cochlear', 'MTF (PCA)', 'MFCC'], 
              [np.mean(c_top), np.mean(m_top), np.mean(mf_top)], 
              yerr=[np.std(c_top), np.std(m_top), np.std(mf_top)], 
              capsize=6, color=['#4dc9f6', '#f67019', '#f53794'], alpha=0.9)

ax.set_ylabel('Pearson Correlation (R)')
ax.set_title('Top 1000 Structural Voxels Prediction Accuracy')
ax.grid(axis='y', linestyle='--', alpha=0.6)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f"{yval:.3f}", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()
